# Notebook 04 — LLM Personalised Health Report Generation & RAGAS Evaluation
### XAI for FPG Prediction: Algorithmic Fairness in Health Insurance Underwriting
**Paper §3.5, §4.6**

This notebook:
1. Converts DiCE counterfactual outputs into structured English prompts
2. Generates personalised health improvement reports via GPT-4o-mini
3. Evaluates report quality using the RAGAS framework
4. Produces one publication-ready figure: `fig_ragas_summary.png` (bar chart, §4.6)

> **Note on language**: Reports are generated in **English** for SSCI submission.  
> The Korean-language version (손보 공모전) is in `part4_llm_report.py`.

> **Prerequisite**: Run `03_dice_counterfactual.ipynb` first.  
> Set `OPENAI_API_KEY` in your `.env` file.


## 1. Setup

In [1]:
import warnings
warnings.filterwarnings("ignore")
import os, json, time, logging
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns

from dotenv import load_dotenv
load_dotenv()

import openai
openai.api_key               = os.getenv("OPENAI_API_KEY")
os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY")

# RAGAS
from ragas import evaluate
from ragas.metrics import (
    faithfulness, answer_relevancy,
    context_recall, context_precision,
)
from ragas.llms       import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from datasets import Dataset

logging.basicConfig(level=logging.INFO,
                    format="%(asctime)s | %(levelname)s | %(message)s")
log = logging.getLogger(__name__)

plt.rcParams.update({
    "font.family":        "DejaVu Sans",
    "font.size":          11,
    "axes.unicode_minus": False,
    "figure.dpi":         150,
})

SEED       = 42
DPI        = 300
MODEL_NAME = "gpt-4o-mini"
OUTPUT_DIR = Path("outputs")
OUTPUT_DIR.mkdir(exist_ok=True)
METRICS    = ["faithfulness","answer_relevancy","context_recall","context_precision"]

log.info("Setup complete. Model: %s", MODEL_NAME)


2026-04-27 15:17:16,390 | INFO | Setup complete. Model: gpt-4o-mini


## 2. Variable Mapping & Group Labels

In [2]:
# ── Group labels ──────────────────────────────────────────────────────────────
GROUP_LABELS = {
    "Young_Male":    "Young Male (19–39 years)",
    "Young_Female":  "Young Female (19–39 years)",
    "Middle_Male":   "Middle-aged Male (40–64 years)",
    "Middle_Female": "Middle-aged Female (40–64 years)",
    "Elderly_Male":  "Elderly Male (65+ years)",
    "Elderly_Female":"Elderly Female (65+ years)",
}

# ── Feature names → readable English labels ────────────────────────────────────
VAR_LABEL = {
    "BMI":               "Body Mass Index",
    "WaistCirc":         "Waist Circumference (cm)",
    "Weight":            "Body Weight (kg)",
    "Energy_kcal":       "Energy Intake (kcal/day)",
    "Carb_g":            "Carbohydrate Intake (g/day)",
    "Sugar_g":           "Sugar Intake (g/day)",
    "Sodium_mg":         "Sodium Intake (mg/day)",
    "Fat_g":             "Fat Intake (g/day)",
    "SatFat_g":          "Saturated Fat Intake (g/day)",
    "Fiber_g":           "Dietary Fibre Intake (g/day)",
    "Potassium_mg":      "Potassium Intake (mg/day)",
    "Protein_g":         "Protein Intake (g/day)",
    "ObesityStatus":     "Obesity Status",
    "SmokingStatus":     "Smoking Status",
    "DrinkingFrequency": "Alcohol Consumption Frequency",
    "DrinkingAmount":    "Alcohol Consumption Amount",
    "AerobicRate":       "Aerobic Physical Activity Rate",
    "ModerateAct_Work":  "Moderate Physical Activity (Occupational)",
    "VigorousAct_Work":  "Vigorous Physical Activity (Occupational)",
    "VigorousAct_Leisure":"Vigorous Physical Activity (Leisure)",
    "WalkingActivity":   "Walking Activity",
    "BreakfastFreq":     "Breakfast Frequency",
    "StressLevel":       "Perceived Stress Level",
    "StressAwareness":   "Stress Awareness Rate",
    "IncomeQuartile":    "Personal Income Quartile",
    "HouseholdIncome":   "Household Income Quartile",
    "EducationLevel":    "Education Level",
    "HealthScreening":   "Health Screening Status",
    "WeightChangeStatus":"Weight Change Status",
    "WeightLossAmount":  "Weight Loss Amount",
    "WeightGainAmount":  "Weight Gain Amount",
}

# ── Group-specific clinical context ───────────────────────────────────────────
GROUP_CLINICAL_CONTEXT = {
    "Young_Male":
        "Young males are particularly susceptible to elevated FPG due to "
        "alcohol consumption, smoking, irregular meal patterns, and sedentary "
        "occupational behaviour. Lifestyle-driven insulin resistance is the "
        "primary mechanism in this group.",
    "Young_Female":
        "Young females often present elevated FPG driven by dietary imbalance "
        "(high refined carbohydrate intake) and low physical activity levels. "
        "Hormonal factors may also influence glycaemic variability.",
    "Middle_Male":
        "Middle-aged males face compounded FPG risk from central adiposity, "
        "habitual alcohol use, and declining insulin sensitivity. "
        "Visceral fat accumulation is the predominant metabolic driver.",
    "Middle_Female":
        "Middle-aged females experience glycaemic instability related to "
        "peri-menopausal hormonal shifts, reduced physical activity, and "
        "psychosocial stress. Dietary quality becomes increasingly influential.",
    "Elderly_Male":
        "Elderly males exhibit FPG elevation driven by sarcopenia (age-related "
        "muscle mass loss), reduced hepatic glucose metabolism, and polypharmacy "
        "effects. Lifestyle modification must account for functional limitations.",
    "Elderly_Female":
        "Elderly females present glycaemic risk amplified by low dietary diversity, "
        "severely reduced physical activity, and osteoporosis-related mobility "
        "restrictions. Social isolation may further limit dietary improvement.",
}

log.info("Variable mappings and group labels defined.")


2026-04-27 15:17:16,411 | INFO | Variable mappings and group labels defined.


## 3. Load DiCE Structured Output (from Notebook 03)

In [3]:
llm_df = pd.read_csv(OUTPUT_DIR / "dice_llm_input.csv", encoding="utf-8")
log.info("Loaded: %d rows × %d cols", *llm_df.shape)
print(llm_df[["pseudo_id","group","source_stage","target_stage",
              "orig_fpg","pred_fpg","top5_changes"]].head(9).to_string(index=False))


2026-04-27 15:17:16,474 | INFO | Loaded: 108 rows × 11 cols


      pseudo_id      group    source_stage  target_stage  orig_fpg  pred_fpg                                                                                                        top5_changes
M-Y-XX-0001-CF1 Young Male Diabetes (≥126) IFG (100–125)     133.0    108.49 {'Sugar_g': 67.115, 'Fiber_g': 25.46, 'DrinkingAmount': 0.667, 'HouseholdIncome': 0.567, 'VigorousAct_Work': 0.333}
M-Y-XX-0001-CF2 Young Male Diabetes (≥126) IFG (100–125)     133.0    102.58 {'Sugar_g': 67.115, 'Fiber_g': 25.46, 'DrinkingAmount': 0.667, 'HouseholdIncome': 0.567, 'VigorousAct_Work': 0.333}
M-Y-XX-0001-CF3 Young Male Diabetes (≥126) IFG (100–125)     133.0    103.68 {'Sugar_g': 67.115, 'Fiber_g': 25.46, 'DrinkingAmount': 0.667, 'HouseholdIncome': 0.567, 'VigorousAct_Work': 0.333}
M-Y-XX-0002-CF1 Young Male Diabetes (≥126) IFG (100–125)     144.0    100.69                     {'Potassium_mg': 4343.256, 'WaistCirc': 5.1, 'ObesityStatus': 1.033, 'BMI': 0.0, 'Weight': 0.0}
M-Y-XX-0002-CF2 Young Male Diabetes

## 4. Prompt Engineering

In [4]:
SYSTEM_PROMPT = """You are a public health AI assistant specialising in
personalised health communication for insurance and preventive medicine contexts.

You convert machine-learning counterfactual analysis outputs (DiCE) into
clear, actionable, personalised health improvement reports in English.

Core principles:
1. Ground all recommendations strictly in the DiCE feature-change data provided.
2. Avoid over-claiming; use hedged language ('predicted', 'estimated', 'suggests').
3. Tailor language to the specific age-sex demographic group.
4. Always end with a mandatory medical disclaimer.
5. Output valid JSON only — no markdown, no code blocks, no preamble."""


def _stage_narrative(source_stage: str, orig_fpg: float) -> str:
    """Clinical risk description appropriate to the FPG stage."""
    if "Diabetes" in source_stage:
        return (f"The current fasting plasma glucose (FPG) of {orig_fpg:.1f} mg/dL "
                f"is in the Diabetes range (≥126 mg/dL). Sustained hyperglycaemia "
                f"at this level is associated with elevated risk of cardiovascular, "
                f"renal, and neuropathic complications.")
    elif "IFG" in source_stage:
        return (f"The current FPG of {orig_fpg:.1f} mg/dL falls in the Impaired "
                f"Fasting Glucose (IFG) range (100–125 mg/dL). Early lifestyle "
                f"intervention at this stage has a high probability of restoring "
                f"normal glycaemic status.")
    return (f"The current FPG of {orig_fpg:.1f} mg/dL is within acceptable "
            f"bounds but warrants preventive lifestyle optimisation.")


def build_user_prompt(row: pd.Series) -> str:
    """
    Convert a single DiCE output row into a fully structured English prompt.

    Includes: pseudonymised ID, group demographics, clinical stage narrative,
    top-5 feature changes with readable labels and direction, and the
    exact JSON schema the model must return.
    """
    # Parse top-5 feature changes
    try:
        changes_raw = eval(row["top5_changes"])
    except Exception:
        changes_raw = {}

    changes_lines = []
    for var, delta in list(changes_raw.items())[:5]:
        label  = VAR_LABEL.get(var, var)
        dirn   = "increase" if delta > 0 else "decrease"
        changes_lines.append(f"  • {label}: {dirn} by {abs(delta):.3f} units")
    changes_str = "\n".join(changes_lines) if changes_lines else "  (no changes identified)"

    pid          = row["pseudo_id"]
    grp_key      = {v:k for k,v in GROUP_LABELS.items()}.get(row["group"], "")
    grp_label    = row["group"]
    clinical_ctx = GROUP_CLINICAL_CONTEXT.get(grp_key, "")
    source_stage = row["source_stage"]
    target_stage = row["target_stage"]
    orig_fpg     = float(row["orig_fpg"])
    pred_fpg     = float(row["pred_fpg"])
    fpg_delta    = round(orig_fpg - pred_fpg, 1)
    stage_narr   = _stage_narrative(source_stage, orig_fpg)

    prompt = f"""Generate a personalised health improvement report for the following case.

══════════════════════════════════════════════════════
PARTICIPANT INFORMATION
  Report ID       : {pid}
  Demographic group: {grp_label}
  Current FPG     : {orig_fpg:.1f} mg/dL  [{source_stage}]
  Target stage    : {target_stage}
  Predicted FPG after lifestyle change: {pred_fpg:.1f} mg/dL
  Estimated FPG reduction: {fpg_delta:.1f} mg/dL

CLINICAL CONTEXT
  {clinical_ctx}
  {stage_narr}

DiCE COUNTERFACTUAL — TOP RECOMMENDED FEATURE CHANGES
{changes_str}
══════════════════════════════════════════════════════

OUTPUT FORMAT — return ONLY this JSON, no other text:
{{
  "report_id": "{pid}",
  "demographic_group": "{grp_label}",
  "executive_summary": "2–3 sentence summary covering current FPG ({orig_fpg:.1f} mg/dL), target stage ({target_stage}), and estimated reduction ({fpg_delta:.1f} mg/dL).",
  "current_status": {{
    "fpg_mgdl": {orig_fpg},
    "stage": "{source_stage}",
    "clinical_risk": "2–3 sentences on health risks for this specific demographic group at this FPG level."
  }},
  "target_status": {{
    "predicted_fpg_mgdl": {pred_fpg},
    "predicted_reduction_mgdl": {fpg_delta},
    "target_stage": "{target_stage}",
    "clinical_significance": "Clinical meaning of a {fpg_delta:.1f} mg/dL reduction for {grp_label}."
  }},
  "personalised_recommendations": [
    {{
      "priority": 1,
      "feature": "Most impactful feature name from DiCE output",
      "direction": "increase or decrease",
      "actionable_change": "Plain-language description of the required change magnitude",
      "rationale": "2 sentences on why this feature affects FPG in {grp_label}.",
      "action_plan": {{
        "short_term": "Concrete action achievable within 1–4 weeks (1 sentence).",
        "long_term":  "Sustained target over 1–3 months (1 sentence)."
      }}
    }},
    {{
      "priority": 2,
      "feature": "Second most impactful feature",
      "direction": "increase or decrease",
      "actionable_change": "Plain-language description",
      "rationale": "2 sentences on relevance to {grp_label}.",
      "action_plan": {{
        "short_term": "1-sentence short-term action.",
        "long_term":  "1-sentence long-term target."
      }}
    }}
  ],
  "group_specific_note": "One additional caution or context specific to {grp_label}.",
  "monitoring_guidance": "Recommended follow-up interval for FPG monitoring (1 sentence).",
  "disclaimer": "This report (ID: {pid}) is generated by an AI model trained on KNHANES population data and does not constitute medical diagnosis or treatment advice. Please consult a qualified healthcare professional before making any changes to your health management plan."
}}"""
    return prompt


log.info("Prompt engineering functions defined.")


2026-04-27 15:17:16,515 | INFO | Prompt engineering functions defined.


## 5. GPT-4o-mini Report Generation

In [5]:
def call_gpt(system_prompt: str, user_prompt: str,
             max_retries: int = 3) -> dict:
    """
    Call GPT API, parse JSON response, retry on failure.
    temperature=0.2 for consistency in medical report generation.
    """
    raw = ""
    for attempt in range(max_retries):
        try:
            response = openai.chat.completions.create(
                model       = MODEL_NAME,
                temperature = 0.2,
                max_tokens  = 1500,
                messages    = [
                    {"role": "system", "content": system_prompt},
                    {"role": "user",   "content": user_prompt},
                ],
            )
            raw = response.choices[0].message.content.strip()
            raw = raw.replace("```json","").replace("```","").strip()
            return json.loads(raw)
        except json.JSONDecodeError as exc:
            log.warning("JSON parse error (attempt %d): %s", attempt+1, exc)
            time.sleep(2)
        except Exception as exc:
            log.warning("API error (attempt %d): %s", attempt+1, exc)
            time.sleep(5)
    return {"error": "Report generation failed", "raw_response": raw}


# ── Generate all reports ──────────────────────────────────────────────────────
reports = []
log.info("Generating %d personalised reports...", len(llm_df))

for idx, row in llm_df.iterrows():
    pid = row["pseudo_id"]
    log.info("[%03d/%d] %s | %s → %s",
             idx+1, len(llm_df), pid, row["source_stage"], row["target_stage"])
    user_prompt = build_user_prompt(row)
    report      = call_gpt(SYSTEM_PROMPT, user_prompt)

    # Attach metadata for downstream processing
    report["_meta"] = {
        "pseudo_id":    pid,
        "group":        row["group"],
        "source_stage": row["source_stage"],
        "target_stage": row["target_stage"],
        "orig_fpg":     float(row["orig_fpg"]),
        "pred_fpg":     float(row["pred_fpg"]),
        "fpg_delta":    float(row["fpg_reduction"]) if "fpg_reduction" in row else
                        round(float(row["orig_fpg"]) - float(row["pred_fpg"]), 2),
    }
    reports.append(report)
    time.sleep(0.5)   # Rate-limit headroom

n_ok  = sum(1 for r in reports if "error" not in r)
n_err = len(reports) - n_ok
log.info("Generation complete: %d OK / %d failed", n_ok, n_err)

# Save
with open(OUTPUT_DIR / "fpg_reports_personalised.json", "w", encoding="utf-8") as f:
    json.dump(reports, f, ensure_ascii=False, indent=2)
log.info("Saved: fpg_reports_personalised.json")


2026-04-27 15:17:16,550 | INFO | Generating 108 personalised reports...
2026-04-27 15:17:16,553 | INFO | [001/108] M-Y-XX-0001-CF1 | Diabetes (≥126) → IFG (100–125)
2026-04-27 15:17:29,166 | INFO | HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2026-04-27 15:17:29,701 | INFO | [002/108] M-Y-XX-0001-CF2 | Diabetes (≥126) → IFG (100–125)
2026-04-27 15:17:40,522 | INFO | HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2026-04-27 15:17:41,031 | INFO | [003/108] M-Y-XX-0001-CF3 | Diabetes (≥126) → IFG (100–125)
2026-04-27 15:17:53,263 | INFO | HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2026-04-27 15:17:53,778 | INFO | [004/108] M-Y-XX-0002-CF1 | Diabetes (≥126) → IFG (100–125)
2026-04-27 15:18:04,395 | INFO | HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2026-04-27 15:18:04,913 | INFO | [005/108] M-Y-XX-0002-CF2 | Diabetes (≥126) → IFG (100–125)
2026-04-27 

## 6. Sample Report Output

In [6]:
sample = next((r for r in reports if "error" not in r), None)
if sample:
    sep = "─" * 60
    print(sep)
    print(f"[Sample Report: {sample.get('report_id','N/A')}]")
    print(f"Group         : {sample.get('demographic_group','')}")
    print(f"Summary       : {sample.get('executive_summary','')}")
    cs = sample.get("current_status", {})
    print(f"Current FPG   : {cs.get('fpg_mgdl')} mg/dL  [{cs.get('stage')}]")
    print(f"Clinical risk : {cs.get('clinical_risk','')}")
    ts = sample.get("target_status", {})
    print(f"Predicted FPG : {ts.get('predicted_fpg_mgdl')} mg/dL  "
          f"(↓{ts.get('predicted_reduction_mgdl')} mg/dL)")
    print(f"Target stage  : {ts.get('target_stage')}")
    for rec in sample.get("personalised_recommendations", []):
        p  = rec.get("priority","?")
        ft = rec.get("feature","")
        d  = rec.get("direction","")
        ac = rec.get("actionable_change","")
        ap = rec.get("action_plan",{})
        print(f"Recommendation {p}: {ft} ({d}) — {ac}")
        print(f"  Short-term: {ap.get('short_term','')}")
        print(f"  Long-term : {ap.get('long_term','')}")
    print(f"Group note    : {sample.get('group_specific_note','')}")
    print(f"Monitoring    : {sample.get('monitoring_guidance','')}")
    print(f"Disclaimer    : {sample.get('disclaimer','')}")
    print(sep)


────────────────────────────────────────────────────────────
[Sample Report: M-Y-XX-0001-CF1]
Group         : Young Male
Summary       : Your current fasting plasma glucose (FPG) is 133.0 mg/dL, which is in the Diabetes range. With lifestyle changes, it is estimated that your FPG could reduce by 24.5 mg/dL, bringing you closer to the target stage of Impaired Fasting Glucose (IFG) at 108.5 mg/dL.
Current FPG   : 133.0 mg/dL  [Diabetes (≥126)]
Clinical risk : At this FPG level, you are at an increased risk for cardiovascular diseases, kidney issues, and nerve damage. Young males with diabetes may also experience complications that can affect overall health and quality of life.
Predicted FPG : 108.49 mg/dL  (↓24.5 mg/dL)
Target stage  : IFG (100–125)
Recommendation 1: Sugar Intake (g/day) (increase) — Increase your daily sugar intake by approximately 67 grams.
  Short-term: Start by reducing sugary snacks and beverages over the next month.
  Long-term : Aim to consistently limit sugar int

## 7. RAGAS Quality Evaluation

In [7]:
# ── Inject LLM and embeddings into RAGAS metrics ────────────────────────────
_llm = LangchainLLMWrapper(ChatOpenAI(model=MODEL_NAME, temperature=0))
_emb = LangchainEmbeddingsWrapper(OpenAIEmbeddings())
faithfulness.llm            = _llm
answer_relevancy.llm        = _llm
answer_relevancy.embeddings = _emb
context_recall.llm          = _llm
context_precision.llm       = _llm


def report_to_answer(report: dict) -> str:
    """Flatten JSON report fields into a single evaluation string for RAGAS."""
    parts = []
    if report.get("executive_summary"):
        parts.append(report["executive_summary"])
    cs = report.get("current_status", {})
    if cs.get("clinical_risk"):
        parts.append(cs["clinical_risk"])
    ts = report.get("target_status", {})
    if ts.get("clinical_significance"):
        parts.append(ts["clinical_significance"])
    for rec in report.get("personalised_recommendations", [])[:3]:
        ft = rec.get("feature","")
        d  = rec.get("direction","")
        rt = rec.get("rationale","")
        ap = rec.get("action_plan",{})
        parts.append(f"{ft} ({d}): {rt} "
                     f"{ap.get('short_term','')} {ap.get('long_term','')}")
    if report.get("group_specific_note"):
        parts.append(report["group_specific_note"])
    return " ".join(parts)


def build_context(row: pd.Series) -> list:
    """Convert DiCE row to RAGAS context string (what the LLM was given)."""
    try:
        changes_raw = eval(row["top5_changes"])
    except Exception:
        changes_raw = {}
    changes_str = ", ".join(
        f"{VAR_LABEL.get(k,k)} ({'increase' if v>0 else 'decrease'} {abs(v):.3f})"
        for k, v in changes_raw.items()
    )
    ctx = (
        f"Report ID: {row['pseudo_id']}, "
        f"Group: {row['group']}, "
        f"Current FPG: {row['orig_fpg']} mg/dL ({row['source_stage']}), "
        f"Target: {row['target_stage']} (predicted {row['pred_fpg']} mg/dL), "
        f"Key feature changes: {changes_str}"
    )
    return [ctx]


def build_ground_truth(row: pd.Series) -> str:
    """Idealised answer grounded in DiCE data + group clinical context."""
    try:
        changes_raw = eval(row["top5_changes"])
    except Exception:
        changes_raw = {}
    top3_actions = ", ".join(
        f"{VAR_LABEL.get(k,k)} {'increase' if v>0 else 'decrease'}"
        for k, v in list(changes_raw.items())[:3]
    )
    grp_key  = {v:k for k,v in GROUP_LABELS.items()}.get(row["group"],"")
    clinical = GROUP_CLINICAL_CONTEXT.get(grp_key,"")
    pid      = row["pseudo_id"]
    return (
        f"[{pid}] The participant ({row['group']}) has a current FPG of "
        f"{row['orig_fpg']} mg/dL ({row['source_stage']}). "
        f"Counterfactual analysis recommends: {top3_actions}. "
        f"{clinical} "
        f"Predicted FPG after changes: {row['pred_fpg']} mg/dL "
        f"({row['target_stage']}). "
        f"AI model output — professional medical consultation required."
    )


# ── Build RAGAS dataset ───────────────────────────────────────────────────────
questions, answers, contexts, ground_truths = [], [], [], []

for report, (_, row) in zip(reports, llm_df.iterrows()):
    if "error" in report:
        log.warning("Skipping %s — report generation failed", row["pseudo_id"])
        continue
    ans = report_to_answer(report)
    if not ans.strip():
        continue
    pid = row["pseudo_id"]
    questions.append(
        f"[{pid}] What are the personalised health improvement recommendations "
        f"for this {row['group']} participant to achieve {row['target_stage']}?"
    )
    answers.append(ans)
    contexts.append(build_context(row))
    ground_truths.append(build_ground_truth(row))
    log.info("%s | answer=%d chars / context=%d chars",
             pid, len(ans), len(contexts[-1][0]))

log.info("RAGAS evaluation dataset: %d samples", len(questions))

eval_dataset = Dataset.from_dict({
    "question"    : questions,
    "answer"      : answers,
    "contexts"    : contexts,
    "ground_truth": ground_truths,
})

ragas_result = evaluate(
    dataset = eval_dataset,
    metrics = [faithfulness, answer_relevancy,
                context_recall, context_precision],
)
df_ragas = ragas_result.to_pandas()
log.info("RAGAS evaluation complete.")
print(df_ragas[METRICS].to_string())


2026-04-27 15:40:31,559 | INFO | M-Y-XX-0001-CF1 | answer=1631 chars / context=395 chars
2026-04-27 15:40:31,561 | INFO | M-Y-XX-0001-CF2 | answer=1738 chars / context=395 chars
2026-04-27 15:40:31,563 | INFO | M-Y-XX-0001-CF3 | answer=1726 chars / context=395 chars
2026-04-27 15:40:31,565 | INFO | M-Y-XX-0002-CF1 | answer=1819 chars / context=350 chars
2026-04-27 15:40:31,567 | INFO | M-Y-XX-0002-CF2 | answer=1797 chars / context=350 chars
2026-04-27 15:40:31,569 | INFO | M-Y-XX-0002-CF3 | answer=1646 chars / context=350 chars
2026-04-27 15:40:31,571 | INFO | M-Y-XX-0003-CF1 | answer=1645 chars / context=349 chars
2026-04-27 15:40:31,573 | INFO | M-Y-XX-0003-CF2 | answer=1640 chars / context=349 chars
2026-04-27 15:40:31,574 | INFO | M-Y-XX-0003-CF3 | answer=1710 chars / context=349 chars
2026-04-27 15:40:31,575 | INFO | M-Y-XX-0001-CF1 | answer=1474 chars / context=356 chars
2026-04-27 15:40:31,576 | INFO | M-Y-XX-0001-CF2 | answer=1653 chars / context=356 chars
2026-04-27 15:40:31,5

     faithfulness  answer_relevancy  context_recall  context_precision
0        0.478261          0.861367            0.75                1.0
1        0.600000          0.841292            0.75                1.0
2        0.280000          0.862085            0.75                1.0
3        0.120000          0.832256            0.50                1.0
4        0.120000          0.877693            0.25                1.0
5        0.120000          0.878182            0.25                1.0
6        0.120000          0.874264            0.50                1.0
7        0.125000          0.865333            0.50                1.0
8        0.120000          0.827290            0.50                1.0
9        0.210526          0.831268            0.50                1.0
10       0.150000          0.827842            0.50                1.0
11       0.111111          0.827854            0.75                1.0
12       0.157895          0.848643            0.75                1.0
13    

## 8. Summary Statistics

In [8]:
summary_records = []
for m in METRICS:
    vals = df_ragas[m].dropna()
    if vals.empty:
        summary_records.append({"Metric":m,"Mean":np.nan,"SD":np.nan,
                                 "Min":np.nan,"Max":np.nan,"Interpretation":"-"})
        continue
    interp = {
        "faithfulness":       ("High variance — some hallucination beyond DiCE bounds"
                               if vals.std() > 0.15 else "Consistent fidelity to DiCE context"),
        "answer_relevancy":   ("Near reference threshold (0.80)" if vals.mean() >= 0.75
                               else "Below reference threshold — prompt revision recommended"),
        "context_recall":     ("Structural ceiling: no medical KB linked"
                               if vals.std() < 0.01 else "Variable coverage"),
        "context_precision":  ("Perfect — structured template effective"
                               if vals.mean() >= 0.99 else "Some irrelevant context included"),
    }.get(m,"")

    summary_records.append({
        "Metric":          m,
        "Mean":            round(vals.mean(), 4),
        "SD":              round(vals.std(),  4),
        "Min":             round(vals.min(),  4),
        "Max":             round(vals.max(),  4),
        "Interpretation":  interp,
    })
    log.info("%-22s  mean=%.4f  SD=%.4f  [%.4f, %.4f]",
             m, vals.mean(), vals.std(), vals.min(), vals.max())

df_summary = pd.DataFrame(summary_records)

# Save
df_ragas.to_csv(OUTPUT_DIR / "table_ragas_per_sample.csv",
                index=False, encoding="utf-8")
df_summary.to_csv(OUTPUT_DIR / "table_ragas_summary.csv",
                  index=False, encoding="utf-8")
print("\n── RAGAS Summary ──")
print(df_summary[["Metric","Mean","SD","Min","Max","Interpretation"]].to_string(index=False))


2026-04-27 15:50:20,157 | INFO | faithfulness            mean=0.2514  SD=0.1776  [0.0769, 1.0000]
2026-04-27 15:50:20,159 | INFO | answer_relevancy        mean=0.8176  SD=0.1404  [0.0000, 0.8823]
2026-04-27 15:50:20,161 | INFO | context_recall          mean=0.5509  SD=0.1701  [0.2500, 0.7500]
2026-04-27 15:50:20,164 | INFO | context_precision       mean=1.0000  SD=0.0000  [1.0000, 1.0000]



── RAGAS Summary ──
           Metric   Mean     SD    Min    Max                                        Interpretation
     faithfulness 0.2514 0.1776 0.0769 1.0000 High variance — some hallucination beyond DiCE bounds
 answer_relevancy 0.8176 0.1404 0.0000 0.8823                       Near reference threshold (0.80)
   context_recall 0.5509 0.1701 0.2500 0.7500                                     Variable coverage
context_precision 1.0000 0.0000 1.0000 1.0000               Perfect — structured template effective


## 9. Figures

> Only `fig_ragas_summary.png` is used in the paper.

In [9]:
# ── Figure: RAGAS metric bar chart — used in paper (§4.6) ───────────────────
def _m(metric): return df_summary.loc[df_summary["Metric"]==metric,"Mean"].values[0]
def _s(metric): return df_summary.loc[df_summary["Metric"]==metric,"SD"].values[0]

METRIC_NICE = {
    "faithfulness":      "Faithfulness\n(DiCE fidelity)",
    "answer_relevancy":  "Answer\nRelevancy",
    "context_recall":    "Context\nRecall",
    "context_precision": "Context\nPrecision",
}
METRIC_COLORS = ["#1565C0","#2E7D32","#C62828","#E65100"]

means  = [_m(m) for m in METRICS]
stds   = [_s(m) for m in METRICS]
labels = [METRIC_NICE[m] for m in METRICS]

fig, ax = plt.subplots(figsize=(10, 6))
bars = ax.bar(labels, means, yerr=stds, capsize=6,
              color=METRIC_COLORS, alpha=0.85,
              edgecolor="white", linewidth=0.8,
              error_kw={"elinewidth":1.8, "ecolor":"#444444", "capthick":1.8})

for bar, mean, std in zip(bars, means, stds):
    if np.isnan(mean): continue
    ax.text(bar.get_x() + bar.get_width()/2,
            bar.get_height() + (std if not np.isnan(std) else 0) + 0.025,
            f"{mean:.4f}", ha="center", va="bottom",
            fontsize=10.5, fontweight="bold")

ax.axhline(0.80, color="gray", ls="--", lw=1.4, alpha=0.7,
           label="Reference threshold (0.80)")
ax.set_ylabel("RAGAS Score (0–1)", fontsize=11)
ax.set_ylim(0, 1.18)
ax.set_title(
    f"LLM Personalised Health Report Quality — RAGAS Evaluation\n"
    f"(Model: {MODEL_NAME}; n = {len(questions)} reports)",
    fontsize=12, fontweight="bold",
)
ax.legend(fontsize=9, loc="upper left")
ax.grid(axis="y", linestyle="--", alpha=0.35)
ax.spines[["top","right"]].set_visible(False)

# Annotate interpretation
annotations = {
    "context_precision": ("Structural ceiling\n(no medical KB)", 0.37),
    "faithfulness":      ("High SD (0.20)\nRAG improvement needed", 0.56),
}
for m_key, (note, y_pos) in annotations.items():
    j = METRICS.index(m_key)
    ax.text(j, y_pos, note, ha="center", va="center",
            fontsize=7.5, color="#666666", style="italic",
            bbox=dict(boxstyle="round,pad=0.25", fc="white", ec="#CCCCCC", alpha=0.85))

plt.tight_layout()
fig.savefig(OUTPUT_DIR / "fig_ragas_summary.png", dpi=DPI, bbox_inches="tight")
plt.show()
log.info("fig_ragas_summary.png saved (paper Figure §4.6).")


2026-04-27 15:50:20,879 | INFO | fig_ragas_summary.png saved (paper Figure §4.6).


## 10. Summary & Paper Text

In [10]:
def _fmt(metric):
    v = _m(metric)
    return f"{v:.4f}" if not np.isnan(v) else "N/A"

print("─" * 60)
print("[Paper §4.6 — Methods text (copy-paste ready)]")
print(f"""
Personalised health improvement reports were generated for all {len(llm_df)}
counterfactual instances (6 demographic groups × 2 transitions × 3 cases
× 3 counterfactuals) using GPT-4o-mini (temperature = 0.2).
Reports were structured via a prompt template encoding the DiCE feature-change
magnitudes, demographic group clinical context, and a mandatory medical
disclaimer. Participants were identified by pseudonymised codes
(e.g., M-Y-XX-0001-CF1) encoding sex, age group, case index, and
counterfactual number. All {n_ok} reports were generated successfully.

Report quality was evaluated using the RAGAS framework (Es et al., 2023),
yielding: Context Precision = {_fmt("context_precision")} (SD = {_s("context_precision"):.3f}),
Answer Relevancy = {_fmt("answer_relevancy")} (SD = {_s("answer_relevancy"):.3f}),
Faithfulness = {_fmt("faithfulness")} (SD = {_s("faithfulness"):.3f}),
and Context Recall = {_fmt("context_recall")} (SD = {_s("context_recall"):.3f}).
Context Precision of 1.000 confirms that the structured prompt template
contains no irrelevant material. The Faithfulness mean of {_fmt("faithfulness")}
with high standard deviation ({_s("faithfulness"):.3f}) reflects cases where
GPT-4o-mini generalises beyond the specific DiCE quantitative bounds —
a known limitation addressable via RAG integration with clinical
knowledge bases (e.g., ADA Standards of Medical Care, 2023).
Context Recall is structurally bounded at {_fmt("context_recall")} because
the prompt supplies only DiCE outputs as context, without a linked
medical knowledge base.""")

print("─" * 60)
log.info("=" * 55)
log.info("Notebook 04 complete.")
for fname in [
    "fpg_reports_personalised.json   — all personalised reports",
    "table_ragas_per_sample.csv      — per-report RAGAS scores",
    "table_ragas_summary.csv         — summary statistics",
    "fig_ragas_summary.png           — paper Figure §4.6 (bar chart)",
]:
    log.info("  outputs/%s", fname)


2026-04-27 15:50:20,906 | INFO | =======================================================
2026-04-27 15:50:20,907 | INFO | Notebook 04 complete.
2026-04-27 15:50:20,908 | INFO |   outputs/fpg_reports_personalised.json   — all personalised reports
2026-04-27 15:50:20,908 | INFO |   outputs/table_ragas_per_sample.csv      — per-report RAGAS scores
2026-04-27 15:50:20,909 | INFO |   outputs/table_ragas_summary.csv         — summary statistics
2026-04-27 15:50:20,910 | INFO |   outputs/fig_ragas_summary.png           — paper Figure §4.6 (bar chart)


────────────────────────────────────────────────────────────
[Paper §4.6 — Methods text (copy-paste ready)]

Personalised health improvement reports were generated for all 108
counterfactual instances (6 demographic groups × 2 transitions × 3 cases
× 3 counterfactuals) using GPT-4o-mini (temperature = 0.2).
Reports were structured via a prompt template encoding the DiCE feature-change
magnitudes, demographic group clinical context, and a mandatory medical
disclaimer. Participants were identified by pseudonymised codes
(e.g., M-Y-XX-0001-CF1) encoding sex, age group, case index, and
counterfactual number. All 108 reports were generated successfully.

Report quality was evaluated using the RAGAS framework (Es et al., 2023),
yielding: Context Precision = 1.0000 (SD = 0.000),
Answer Relevancy = 0.8176 (SD = 0.140),
Faithfulness = 0.2514 (SD = 0.178),
and Context Recall = 0.5509 (SD = 0.170).
Context Precision of 1.000 confirms that the structured prompt template
contains no irrelevant mate